# 01 · Base de conhecimento em trechos

**Tarefa da IA:** responder dúvidas de passageiros de ônibus de São Paulo (idosos, pessoas com deficiência, estudantes) sobre gratuidade, Bilhete Único, integração, tarifas e direitos no transporte, **citando a fonte** e **recusando** quando a base não tem a resposta.

Fontes: só documentos públicos oficiais (SPTrans e leis federais). As leis vêm do portal da Câmara dos Deputados porque o site do Planalto não respondeu a requisições automáticas durante a coleta (ver `scripts/baixar_leis.py`).

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src'))
import pandas as pd
pd.set_option('display.max_colwidth', 120)

In [2]:
fontes = pd.read_csv(RAIZ / 'base' / 'fontes.csv')
fontes[['id', 'orgao', 'formato', 'licenca', 'data_acesso']]

,id,orgao,formato,licenca,data_acesso
0,sptrans-gratuidade-65-mais,SPTrans - São Paulo Transporte S.A.,html,Conteúdo público institucional; uso acadêmico com citação da fonte,2026-11-11
1,sptrans-gratuidade-60-64,SPTrans - São Paulo Transporte S.A.,html,Conteúdo público institucional; uso acadêmico com citação da fonte,2026-11-11
2,sptrans-bilhete-pcd,SPTrans - São Paulo Transporte S.A.,html,Conteúdo público institucional; uso acadêmico com citação da fonte,2026-11-11
3,sptrans-bilhete-unico,SPTrans - São Paulo Transporte S.A.,html,Conteúdo público institucional; uso acadêmico com citação da fonte,2026-11-11
4,sptrans-bilhete-estudante,SPTrans - São Paulo Transporte S.A.,html,Conteúdo público institucional; uso acadêmico com citação da fonte,2026-11-11
5,sptrans-tarifas,SPTrans - São Paulo Transporte S.A.,html,Conteúdo público institucional; uso acadêmico com citação da fonte,2026-11-11
6,lei-10741-transporte,Câmara dos Deputados (texto oficial da lei federal),HTML,"Domínio público (Lei 9.610/98, art. 8º, IV: textos de lei não são protegidos)",2026-11-11
7,lei-13146-transporte,Câmara dos Deputados (texto oficial da lei federal),HTML,"Domínio público (Lei 9.610/98, art. 8º, IV: textos de lei não são protegidos)",2026-11-11


## Divisão em trechos
- **Leis:** um trecho por artigo (unidade natural de citação); artigos longos são quebrados por parágrafo mantendo o rótulo `Art. X (cont.)`.
- **Perguntas e Respostas da SPTrans:** um trecho por par pergunta + resposta (cada FAQ já responde uma dúvida completa).
- **Tabela de tarifas:** cada célula vira uma frase (`Tarifa ...: R$ 5,30.`) sem alterar nenhum valor.
- Limite de ~700 caracteres por trecho: cabe no contexto do modelo de 3B junto com outros 3 trechos.

In [3]:
from pf_genai import trechos
T = trechos.gerar_trechos()
trechos.salvar(T)
print(len(T), 'trechos gerados')

244 trechos gerados


In [4]:
df = pd.DataFrame([t.__dict__ for t in T])
tabela = (df.groupby('documento_id').agg(trechos=('trecho_id', 'count'), caracteres_medios=('texto', lambda s: int(s.str.len().mean())))
          .join(fontes.set_index('id')[['titulo', 'formato', 'licenca']]))
tabela

,trechos,caracteres_medios,titulo,formato,licenca
documento_id,,,,,
lei-10741-transporte,6,402,Lei 10.741/2003 (Estatuto da Pessoa Idosa) · Do Transporte,HTML,"Domínio público (Lei 9.610/98, art. 8º, IV: textos de lei não são protegidos)"
lei-13146-transporte,12,410,Lei 13.146/2015 (Lei Brasileira de Inclusão) · Do Direito ao Transporte e à Mobilidade,HTML,"Domínio público (Lei 9.610/98, art. 8º, IV: textos de lei não são protegidos)"
sptrans-bilhete-estudante,47,416,Perguntas e Respostas SPTrans - Bilhete Único de Estudante,html,Conteúdo público institucional; uso acadêmico com citação da fonte
sptrans-bilhete-pcd,60,425,Perguntas e Respostas SPTrans - Bilhete Único Especial da Pessoa com Deficiência,html,Conteúdo público institucional; uso acadêmico com citação da fonte
sptrans-bilhete-unico,52,327,"Perguntas e Respostas SPTrans - Bilhete Único (tipos, integração, uso)",html,Conteúdo público institucional; uso acadêmico com citação da fonte
sptrans-gratuidade-60-64,33,334,Perguntas e Respostas SPTrans - Gratuidade para pessoas com 60 a 64 anos,html,Conteúdo público institucional; uso acadêmico com citação da fonte
sptrans-gratuidade-65-mais,28,470,Perguntas e Respostas SPTrans - Bilhete Único Especial da Pessoa Idosa (65+),html,Conteúdo público institucional; uso acadêmico com citação da fonte
sptrans-tarifas,6,584,Tarifas vigentes - SPTrans,html,Conteúdo público institucional; uso acadêmico com citação da fonte


In [5]:
tabela[['titulo', 'formato', 'licenca', 'trechos']].to_csv(RAIZ / 'base' / 'tabela_base.csv')
print(tabela['trechos'].sum(), 'trechos no total')

244 trechos no total


## Exemplos de trechos

In [6]:
for tid in ['lei-10741-transporte-002', 'sptrans-gratuidade-60-64-025', 'sptrans-tarifas-003']:
    t = next(x for x in T if x.trecho_id == tid)
    print(f'[{t.trecho_id}] ({t.documento})\n{t.texto}\n')

[lei-10741-transporte-002] (Lei 10.741/2003 (Estatuto da Pessoa Idosa) · Do Transporte)
Art. 39. Aos maiores de 65 (sessenta e cinco) anos fica assegurada a gratuidade dos transportes coletivos públicos urbanos e semi-urbanos, exceto nos serviços seletivos e especiais, quando prestados paralelamente aos serviços regulares.
§ 1º Para ter acesso à gratuidade, basta que a pessoa idosa apresente qualquer documento pessoal que faça prova de sua idade. (Parágrafo com redação dada pela Lei nº 14.423, de 22/7/2022)

[sptrans-gratuidade-60-64-025] (Perguntas e Respostas SPTrans - Gratuidade para pessoas com 60 a 64 anos)
Qual é o prazo para fazer a primeira recarga da gratuidade no Bilhete Único Comum, após a pessoa completar 60 anos de idade?
A recarga da gratuidade estará disponível em 72 horas após a data de aniversário. Basta aproximar o seu cartão em qualquer um dos equipamentos de recargas disponíveis nas estações do metrô/trem, nos ônibus ou terminais municipais. Nesse intervalo, a pesso